In [1]:
import cv2 as cv
import threading
import random
import time
import ipywidgets as widgets
from IPython.display import display
from positioning import color_follow
from jetcobot_utils.jetcobot_config import *

In [2]:
follow = color_follow()
model = 'General'
HSV_learning = ()
color_hsv = {"red": ((0, 25, 90), (10, 255, 255)),
             "green": ((53, 36, 40), (80, 255, 255)),
             "blue": ((110, 80, 90), (120, 255, 255)),
             "yellow": ((25, 20, 55), (50, 255, 255))}
color = [[random.randint(0, 255) for _ in range(3)] for _ in range(255)]
HSV_path="/home/jetson/jetcobot_ws/src/jetcobot_color_identify/scripts/HSV_config.txt"

try: read_HSV(HSV_path,color_hsv)
except Exception: print("Read HSV_config Error !!!")

In [3]:
button_layout = widgets.Layout(width='200px', height='100px', align_self='center')
# 输出控件 Output widget
output = widgets.Output()
# 颜色追踪 Color tracking
color_follow = widgets.Button(description='Start', button_style='success', layout=button_layout)
# 选择颜色 Select color
choose_color = widgets.ToggleButtons(options=['red', 'green', 'blue', 'yellow'], button_style='success',
             tooltips=['Description of slow', 'Description of regular', 'Description of fast'])
# 取消追踪 Cancel tracking
follow_cancel = widgets.Button(description='Cancel', button_style='danger', layout=button_layout)

# 退出 exit
exit_button = widgets.Button(description='Exit', button_style='danger', layout=button_layout)
# 图像控件 Image widget
imgbox = widgets.Image(format='jpg', height=480, width=640, layout=widgets.Layout(align_self='auto'))
# 垂直布局 Vertical layout
img_box = widgets.VBox([imgbox, choose_color], layout=widgets.Layout(align_self='auto'))
# 垂直布局 Vertical layout
Slider_box = widgets.VBox([color_follow,follow_cancel,exit_button],
                          layout=widgets.Layout(align_self='auto'))
# 水平布局 Horizontal layout
controls_box = widgets.HBox([img_box, Slider_box], layout=widgets.Layout(align_self='auto'))
# ['auto', 'flex-start', 'flex-end', 'center', 'baseline', 'stretch', 'inherit', 'initial', 'unset']

In [4]:
def color_follow_Callback(value):
    global model
    model = 'color_follow'

def follow_cancel_Callback(value):
    global model
    model = 'General'

def exit_button_Callback(value):
    global model
    model = 'Exit'

color_follow.on_click(color_follow_Callback)
follow_cancel.on_click(follow_cancel_Callback)
exit_button.on_click(exit_button_Callback)

In [5]:
def camera():
    global HSV_learning,model
    # 打开摄像头 Open camera
    capture = cv.VideoCapture(0)
    capture.set(3, 640)
    capture.set(4, 480)
    capture.set(5, 30)  
    m_fps = 0
    t_start = time.time()
    # Be executed in loop when the camera is opened normally 
    # 当摄像头正常打开的情况下循环执行
    while capture.isOpened():
        try:
            _, img = capture.read()
            if model == 'color_follow':
                img = follow.follow_function(img, color_hsv[choose_color.value])
                cv.putText(img, choose_color.value, (int(img.shape[0] / 2), 50), cv.FONT_HERSHEY_SIMPLEX, 2, color[random.randint(0, 254)], 2)
           
            if model == 'learning_color':
                img,HSV_learning = follow.get_hsv(img)
                
            if model == 'Exit':
                cv.destroyAllWindows()
                capture.release()
                break
            m_fps = m_fps + 1
            fps = m_fps / (time.time() - t_start)
            if (time.time() - t_start) >= 2:
                m_fps = fps
                t_start = time.time() - 1
            text="FPS:" + str(int(fps))
            cv.putText(img, text, (10, 20), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
            imgbox.value = cv.imencode('.jpg', img)[1].tobytes()
        except:
            capture.release()

In [6]:
display(controls_box,output)
threading.Thread(target=camera, ).start()

Output()

[ WARN:0@2.183] global cap_gstreamer.cpp:1777 open OpenCV | GStreamer warning: Cannot query video position: status=0, value=-1, duration=-1
